# Lesson 2.4 · When tool calls go wrong: bad arguments, malformed calls, recovery

**Goal:** turn tool failures into results the model can recover from, and tell the two kinds of failure apart.

**What you will be able to do**
- Read a malformed tool call on the wire (a 400 before any tool runs) and answer it with a nudge, not a blind retry.
- Catch a tool's exception and send it back as a tool result with the call's id, so the loop keeps going.
- Write a helpful error (what went wrong + what is valid): in the video it took correct data from 0/8 to 8/8.
- Make writes fail closed: refuse a guessed argument, never act on it.

> One cell **calls the API** (3 policies × 3 runs, small prompts), plus one optional cell in exercise 1. Replies are saved in `./cache`, so re-runs are free and work without a key.

In [1]:
# Setup: run this cell first.
# It finds the course's nightwatch/ folder (the Brightcart simulator), loads GROQ_API_KEY from .env if there is one,
# and builds `chat(...)`: a client call with a record/replay cache in ./cache, so re-runs cost nothing.
import hashlib, json, os, sys
from pathlib import Path

HERE = Path.cwd()
NW = next((p / "nightwatch" for p in [HERE, *HERE.parents] if (p / "nightwatch" / "brightcart").is_dir()), None)
assert NW is not None, "Can't find nightwatch/. Open this notebook from inside the course folder (lessons/<lesson>/notebook)."
sys.path.insert(0, str(NW))
from brightcart.incidents import fresh, generate
from brightcart.sim import World

try:
    from dotenv import load_dotenv
    load_dotenv(NW.parent / ".env")
except ImportError:
    pass
KEY = os.environ.get("GROQ_API_KEY")

import httpx
from openai import BadRequestError, OpenAI
from openai.types.chat import ChatCompletion

client = OpenAI(api_key=KEY or "missing", base_url="https://api.groq.com/openai/v1", max_retries=8)
MODEL = "openai/gpt-oss-20b"
CACHE = HERE / "cache"


def chat(sample=0, **body):
    """One model call with record/replay: a cached reply is replayed for free, a new one calls Groq once and is saved.
    A 400 (malformed tool call) is cached too and raised again as BadRequestError, like the real client does."""
    h = hashlib.sha256((json.dumps(body, sort_keys=True, default=str) + f"#{sample}").encode()).hexdigest()[:20]
    f = CACHE / f"{h}.json"
    if f.exists():
        saved = json.loads(f.read_text(encoding="utf-8"))
        if "error_400" in saved:
            raise BadRequestError(saved["error_400"], body=None,
                                  response=httpx.Response(400, request=httpx.Request("POST", str(client.base_url))))
        saved.pop("service_tier", None)            # Groq sends "on_demand", which the SDK type does not list
        return ChatCompletion.model_validate(saved)
    if not KEY:
        raise RuntimeError("This reply is not in ./cache and GROQ_API_KEY is not set. Get the free key (lesson 0.2), "
                           "put GROQ_API_KEY=... in the course's .env, and re-run the setup cell.")
    CACHE.mkdir(exist_ok=True)
    try:
        r = client.chat.completions.create(**body)
    except BadRequestError as e:
        f.write_text(json.dumps({"error_400": str(e)}), encoding="utf-8")
        raise
    f.write_text(r.model_dump_json(), encoding="utf-8")
    return r


def check(n, test):
    """Run an exercise check; print a clear line instead of stopping the notebook."""
    try:
        test()
        print(f"✓ Exercise {n} passed")
    except Exception as e:
        print(f"✗ Exercise {n} not yet: {type(e).__name__}: {str(e) or 'an assert failed, see the check cell'}")


print("nightwatch found:", NW.name, "| GROQ_API_KEY:", "set" if KEY else
      "NOT set -> cells marked 'calls the API' replay from ./cache; a new question needs the free key (lesson 0.2)")

nightwatch found: nightwatch | GROQ_API_KEY: set


## 1. Recall: one call, one result (lesson 2.3)

With several tool calls in one reply, each call got exactly one tool message with its id. Today that one result is an **error**, and it still has to go back.

In [2]:
calls = ["id_a", "id_b", "id_c"]
answers = [{"role": "tool", "tool_call_id": i, "content": "..."} for i in calls]
print("every id answered once:", sorted(t["tool_call_id"] for t in answers) == sorted(calls))

every id answered once: True


## 2. Key words

| Word | Meaning |
|---|---|
| bad arguments | valid JSON with a wrong value, like an unknown service name |
| exception | how Python stops a function and reports what went wrong |
| error as a tool result | the error text goes back as the tool's answer |
| malformed tool call | arguments that are not even valid JSON |
| 400 | status code for a bad request: the whole request is rejected |
| recovery | the model fixes its own call and still reaches the data |
| nudge | a short message saying what went wrong, so the next request is different |

## 3. The idea: two kinds of failure, three policies

**Malformed call.** Groq checks the arguments first; if they are not valid JSON the whole request fails with a 400, *Failed to parse tool call arguments as JSON*. No tool call reaches your code. Sending the same request again at temperature 0 gives the same failure (4 of 4 in the lesson 2.3 run). A **nudge** changes the request, so the model gets a real second chance.

**Bad argument.** The JSON is fine, the value is wrong: `payment`, not `payments`.

| policy | what your code does on the KeyError | what goes back |
|---|---|---|
| crash | nothing catches it | the run stops |
| vague | catch it | `"Error"` |
| helpful | catch it | `ERROR: "unknown service 'payment'; known: web, checkout, payments, ..."` |

### Hand trace (the video's run, 8 runs per policy)

| policy | correct data | what the model tried |
|---|---|---|
| crash | 0/8 | `['payment']`, then the crash, 8 times |
| vague | 1/8 | mostly `['payment', 'payment', ...]`, once `'payment-service'`, once `['payment', 'payments']`; 7 runs answered **without data** |
| helpful | 8/8 | `['payment', 'payments']` in all 8 runs |

Only the error text changed, and the score went from 0 to 8.

## 4. Read the wire, then write it yourself

### 4.1 A malformed call (saved run from lesson 2.3)

In [3]:
ev_file = HERE.parent / "examples" / "evidence_blind_retry_from_2_3.txt"
if ev_file.exists():
    wire = [l.strip() for l in ev_file.read_text(encoding="utf-8").splitlines() if "400 on turn" in l]
else:                                           # outside the course folder: the same four lines
    wire = ["[openai/gpt-oss-20b] 400 on turn 4: Failed to parse tool call arguments as JSON -> retry"] * 4
for l in wire:
    print(" ", l)
print("  -> same request, temperature 0, four retries, four identical failures")

  [openai/gpt-oss-20b] 400 on turn 4: Failed to parse tool call arguments as JSON -> retry
  [openai/gpt-oss-20b] 400 on turn 4: Failed to parse tool call arguments as JSON -> retry
  [openai/gpt-oss-20b] 400 on turn 4: Failed to parse tool call arguments as JSON -> retry
  [openai/gpt-oss-20b] 400 on turn 4: Failed to parse tool call arguments as JSON -> retry
  -> same request, temperature 0, four retries, four identical failures


Why a blind retry can't help: the request is byte-for-byte the same, so (at temperature 0) is the answer. A nudge makes it a different request.

In [4]:
def fingerprint(body):
    return hashlib.sha256(json.dumps(body, sort_keys=True).encode()).hexdigest()[:12]

body = {"model": MODEL, "temperature": 0, "messages": [{"role": "user", "content": "How is payments doing?"}]}
retry = json.loads(json.dumps(body))
nudged = json.loads(json.dumps(body))
nudged["messages"].append({"role": "user", "content": "Your tool call was malformed JSON. Try again."})
print("original:", fingerprint(body), "| blind retry:", fingerprint(retry), "| nudged:", fingerprint(nudged))

original: 40ae36722d81 | blind retry: 40ae36722d81 | nudged: 7f2aefe0b543


### 4.2 A helpful error starts in the tool

In [5]:
world = World(fresh(next(x for x in generate() if x.id == "upstream_deploy-0")))
TOOLS = [{"type": "function", "function": {
    "name": "get_metrics", "description": "Error %, p95 latency, CPU and memory of one service.",
    "parameters": {"type": "object", "properties": {"service": {"type": "string"}}, "required": ["service"]}}}]
QUESTION = "How is the payment service doing? Use get_metrics, then answer in one sentence."


def get_metrics(service):
    return world.status(service)        # KeyError for an unknown name, listing the known ones


print("real names:", ", ".join(world.inc.services))
try:
    get_metrics("payment-service")
except KeyError as e:
    print("ERROR:", e)

real names: web, checkout, payments, inventory, search, email-worker
ERROR: "unknown service 'payment-service'; known: web, checkout, payments, inventory, search, email-worker"


### 4.3 try / except around every tool call, nudge around every model call

In [6]:
def run(policy, max_turns=5, sample=0):
    "policy: crash | vague | helpful, or a function(exception) -> error text. Returns (outcome, names tried)."
    msgs = [{"role": "user", "content": QUESTION}]
    tried = []
    for _ in range(max_turns):
        try:
            r = chat(sample=sample, model=MODEL, temperature=1, messages=msgs, tools=TOOLS)
        except BadRequestError:                                  # malformed call: nudge, don't resend the same
            msgs.append({"role": "user", "content": "Your tool call was malformed JSON. Try again."})
            continue
        m = r.choices[0].message
        if not m.tool_calls:
            return ("answered" if "payments" in tried else "answered without data"), tried
        msgs.append({"role": "assistant", "content": m.content or "", "tool_calls": [c.model_dump() for c in m.tool_calls]})
        for c in m.tool_calls:
            args = json.loads(c.function.arguments or "{}")
            tried.append(args.get("service"))
            try:
                out = json.dumps(get_metrics(**args))
            except Exception as e:
                if policy == "crash":
                    return f"crashed: {type(e).__name__}", tried
                out = "Error" if policy == "vague" else policy(e) if callable(policy) else f"ERROR: {e}"
            msgs.append({"role": "tool", "tool_call_id": c.id, "content": out})   # exactly one result per id
    return "gave up", tried

### 4.4 The race: three policies, 3 runs each (temperature 1, so your numbers may differ)

In [7]:
# calls the API (replays from ./cache when the reply is saved)
N = 3
for policy in ("crash", "vague", "helpful"):
    outcomes = [run(policy, sample=i) for i in range(N)]
    ok = sum(o == "answered" for o, _ in outcomes)
    print(f"\npolicy {policy:8}: correct data reached {ok}/{N}")
    for o, t in outcomes:
        print(f"   {o:24} tried {t}")


policy crash   : correct data reached 0/3
   crashed: KeyError        tried ['payment']
   crashed: KeyError        tried ['payment']
   crashed: KeyError        tried ['payment']



policy vague   : correct data reached 0/3
   answered without data    tried ['payment']
   answered without data    tried ['payment']
   answered without data    tried ['payment', 'payment']



policy helpful : correct data reached 3/3
   answered                 tried ['payment', 'payments']
   answered                 tried ['payment', 'payments']
   answered                 tried ['payment', 'payments']


Crash dies on the first wrong name. Vague lets the model guess the same wrong name again and then answer anyway: those answers are made up. Helpful names the valid choices, and the model fixes its call.

## 5. The library version + check: Nightwatch's wrapper

Nightwatch (`agent_v0.run`) wraps every tool the same way: `try: run_tool(...)` / `except Exception as e: result = f"ERROR: {e}"`. Its client also spots the malformed-call 400 by the code `tool_use_failed` and hands it back to the loop instead of crashing. Check: its error text is exactly our helpful text.

In [8]:
import inspect
import agent_v0

try:
    agent_v0.run_tool(world, "service_status", {"service": "payment"})
except Exception as e:
    theirs = f"ERROR: {e}"
try:
    get_metrics("payment")
except KeyError as e:
    ours = f"ERROR: {e}"
print(theirs)
print("same text as our helpful policy:", ours == theirs)
print("client handles:", [l.strip() for l in inspect.getsource(agent_v0.chat).splitlines() if "tool_use_failed" in l][0])

ERROR: "unknown service 'payment'; known: web, checkout, payments, inventory, search, email-worker"
same text as our helpful policy: True
client handles: if e.code == 400 and ("tool_use_failed" in msg or "output_parse_failed" in msg):


## 6. Where this shows up in agents: reads may retry, writes fail closed

For a **read**, a wrong guess costs nothing, so letting the model try again is fine. A **write** is different: `restart_service("payment")` must **fail closed**, refuse and report, never guess "you probably meant payments". The simulator already refuses unknown targets; your tool should refuse before it even gets there.

In [9]:
try:
    world.act("restart_service", "payment")
except KeyError as e:
    print("world refuses:", e, "| writes so far:", world.actions)

world refuses: "unknown target 'payment'" | writes so far: []


## 7. Traps (each one runs)

**Trap 1: letting the exception escape the loop.** One bad name ends the whole run.

In [10]:
try:
    for name in ["payment", "payments"]:        # the second, correct call never happens
        print(get_metrics(name))
except KeyError as e:
    print("run over after one bad name:", e)

run over after one bad name: "unknown service 'payment'; known: web, checkout, payments, inventory, search, email-worker"


**Trap 2: sending back just the word "Error".** The model learns nothing, so it guesses again (vague: 1/8 in the video).

In [11]:
for text in ["Error", ours]:
    print(f"{text[:60]:62} mentions the real name: {'payments' in text}")

Error                                                          mentions the real name: False
ERROR: "unknown service 'payment'; known: web, checkout, pay   mentions the real name: True


**Trap 3: retrying a malformed call blindly.** Same request, same failure: see the fingerprints in 4.1.

**Trap 4: trusting an answer that came after an error.** Check that some tool really returned the data.

In [12]:
tool_results = ["Error", "Error"]               # what a vague run sent back
answer = "The payment service is healthy with low error rates."
try:
    assert any(not t.startswith(("Error", "ERROR")) for t in tool_results), "no tool ever returned data"
    print("answer backed by data:", answer)
except AssertionError as e:
    print("AssertionError:", e, "-> don't trust:", answer)

AssertionError: no tool ever returned data -> don't trust: The payment service is healthy with low error rates.


**Trap 5: letting the model retry a write with a guessed argument.** "Close enough" is not enough for a write.

In [13]:
import difflib
guess = difflib.get_close_matches("payment", list(world.inc.services), n=1)
print("a fuzzy matcher would restart:", guess, "-> for a write: refuse instead; the name must match exactly")

a fuzzy matcher would restart: ['payments'] -> for a write: refuse instead; the name must match exactly


## 8. Exercises

### Exercise 1 (the video's exercise, step 1)
Cut the known names from the error. Write `bare_error(e)`: it gets the `KeyError` and returns `"ERROR: unknown service '<name>'"` with **no** list of valid names. (`e.args[0]` is the full message.) The optional API cell races it like 4.4, to compare with helpful 8/8.

In [14]:
def bare_error(e):
    # YOUR CODE HERE
    return ""

In [15]:
# calls the API (replays from ./cache when the reply is saved)
# optional: race the bare error (3 runs; skipped while bare_error is unfinished)
try:
    get_metrics("payment")
except KeyError as e:
    ready = bool(bare_error(e))
if ready:
    outcomes = [run(bare_error, sample=i) for i in range(3)]
    print("bare error: correct data reached", sum(o == "answered" for o, _ in outcomes), "/ 3")
    for o, t in outcomes:
        print(f"   {o:24} tried {t}")

In [16]:
def _t1():
    try:
        get_metrics("payment-service")
    except KeyError as e:
        text = bare_error(e)
    assert text == "ERROR: unknown service 'payment-service'", text
    assert not any(s in text for s in world.inc.services if s != "payment-service"), "no valid names in the bare error"
check(1, _t1)

✗ Exercise 1 not yet: AssertionError: an assert failed, see the check cell


### Exercise 2 (the video's exercise, step 2): a restart that fails closed
Write `restart_service(service)`: if `service` is exactly one of `world.inc.services`, call `world.act("restart_service", service)` and return its text; otherwise return `"REFUSED: unknown service '<name>'; known: ..."` and do **not** call `world.act`.

In [17]:
def restart_service(service):
    # YOUR CODE HERE
    pass

In [18]:
def _t2():
    start = len(world.actions)
    for guess in ["payment", "Payments", "payment-service", ""]:
        out = restart_service(guess)
        assert str(out).startswith("REFUSED:") and "payments" in str(out), f"{guess!r} -> {out!r}"
    assert len(world.actions) == start, "a guessed name must never reach world.act"
    assert "done" in restart_service("payments") and world.actions[-1] == ("restart_service", "payments", None)
check(2, _t2)

✗ Exercise 2 not yet: AssertionError: 'payment' -> None


### Exercise 3: one wrapper for every tool call
Write `tool_result(call, funcs)`: `call` is a dict `{"id", "function": {"name", "arguments"}}`, `funcs` maps names to functions. Always return **one** tool message with the call's id. Its content is the JSON result, or an `ERROR: ...` text when the arguments are not valid JSON, the tool is unknown (list the known tools), or the tool raises.

In [19]:
def tool_result(call, funcs):
    # YOUR CODE HERE
    return None

In [20]:
def _t3():
    funcs = {"get_metrics": get_metrics}
    mk = lambda i, n, a: {"id": i, "function": {"name": n, "arguments": a}}
    ok = tool_result(mk("c1", "get_metrics", '{"service": "payments"}'), funcs)
    assert ok["role"] == "tool" and ok["tool_call_id"] == "c1" and json.loads(ok["content"]) == world.status("payments")
    bad = tool_result(mk("c2", "get_metrics", '{"service": "payment"}'), funcs)
    assert bad["tool_call_id"] == "c2" and bad["content"].startswith("ERROR") and "payments" in bad["content"]
    broken = tool_result(mk("c3", "get_metrics", '{"service": "payme'), funcs)
    assert broken["tool_call_id"] == "c3" and broken["content"].startswith("ERROR"), "malformed JSON"
    unknown = tool_result(mk("c4", "restart_all", "{}"), funcs)
    assert unknown["content"].startswith("ERROR") and "get_metrics" in unknown["content"], "list the known tools"
check(3, _t3)

✗ Exercise 3 not yet: TypeError: 'NoneType' object is not subscriptable


### Exercise 4: nudge, don't resend
Write `next_request(body)`: return a **new** request body (leave `body` unchanged) with one extra user message that says the last tool call was malformed JSON and asks for one tool call with valid JSON.

In [21]:
def next_request(body):
    # YOUR CODE HERE
    return body

In [22]:
def _t4():
    before = fingerprint(body)
    new = next_request(body)
    assert fingerprint(body) == before, "don't change the original body"
    assert fingerprint(new) != before, "the new request must differ from a blind retry"
    assert len(new["messages"]) == len(body["messages"]) + 1 and new["messages"][-1]["role"] == "user"
    assert "json" in new["messages"][-1]["content"].lower(), "say what went wrong"
check(4, _t4)

✗ Exercise 4 not yet: AssertionError: the new request must differ from a blind retry


## Solutions (try first!)

In [23]:
def bare_error(e):
    bad = e.args[0].split("'")[1]          # "unknown service 'payment-service'; known: ..." -> payment-service
    return f"ERROR: unknown service '{bad}'"

In [24]:
def restart_service(service):
    known = list(world.inc.services)
    if service not in known:
        return f"REFUSED: unknown service {service!r}; known: {', '.join(known)}"
    return world.act("restart_service", service)

In [25]:
def tool_result(call, funcs):
    name = call["function"]["name"]
    try:
        args = json.loads(call["function"]["arguments"] or "{}")
        if name not in funcs:
            raise KeyError(f"unknown tool {name!r}; known: {', '.join(funcs)}")
        content = json.dumps(funcs[name](**args), default=str)
    except json.JSONDecodeError as e:
        content = f"ERROR: arguments are not valid JSON ({e.msg}); send them again as one JSON object"
    except Exception as e:
        content = f"ERROR: {e}"
    return {"role": "tool", "tool_call_id": call["id"], "content": content}

In [26]:
def next_request(body):
    new = json.loads(json.dumps(body))
    new["messages"].append({"role": "user", "content": "Your last tool call was malformed JSON. "
                                                       "Call one tool again with valid JSON arguments."})
    return new